In [ ]:
!pip install emoji
!pip install nltk

In [ ]:
import re
import string
import emoji
import pandas as pd
import nltk

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

In [ ]:
df = pd.read_csv('twitter_training.csv')

In [ ]:
# ---- One-time NLTK setup (run once) ----
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
lemmatizer = WordNetLemmatizer()

# Precompile regex patterns (FASTER)
URL_PATTERN = re.compile(r'http\S+|www\S+')
MENTION_PATTERN = re.compile(r'@\w+')
HASHTAG_PATTERN = re.compile(r'#')
REPEAT_PATTERN = re.compile(r'(.)\1{2,}')
RT_PATTERN = re.compile(r'\brt\b')
SPECIAL_CHAR_PATTERN = re.compile(r"[^a-zA-Z0-9\s!?.,']")
MULTISPACE_PATTERN = re.compile(r'\s+')

# Preload stopwords ONCE
STOP_WORDS = set(stopwords.words('english')) - {'not', 'no', 'but', 'however', 'yet'}

# Translation table for punctuation
PUNCT_TABLE = str.maketrans('', '', string.punctuation)

# WordNet POS Mapper

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

# Vectorized Text Cleaning

def vectorized_clean_text(series):
    series = series.str.lower()
    series = series.str.replace(URL_PATTERN, '', regex=True)
    series = series.str.replace(MENTION_PATTERN, '', regex=True)
    series = series.str.replace(HASHTAG_PATTERN, '', regex=True)
    series = series.apply(emoji.demojize)
    series = series.str.replace(REPEAT_PATTERN, r'\1\1', regex=True)
    series = series.str.replace(RT_PATTERN, '', regex=True)
    series = series.str.replace(SPECIAL_CHAR_PATTERN, '', regex=True)
    series = series.str.replace(MULTISPACE_PATTERN, ' ', regex=True)
    series = series.str.strip()
    return series

# Remove Punctuation (Vectorized)

def remove_punctuation_series(series):
    return series.str.translate(PUNCT_TABLE)

# Remove Stopwords (Single Pass)

def remove_stopwords_series(series):
    return series.apply(
        lambda text: " ".join(
            word for word in text.split()
            if word not in STOP_WORDS
        )
    )

# POS-aware Lemmatization

def lemmatize_series(series):
    def lemmatize_text(text):
        words = text.split()
        tagged_words = pos_tag(words)
        lemmas = [
            lemmatizer.lemmatize(word, get_wordnet_pos(tag))
            for word, tag in tagged_words
        ]
        return " ".join(lemmas)

    return series.apply(lemmatize_text)

def optimized_preprocessing_pipeline(df):

    # Structural cleaning
    df = df.drop(columns=['2401', 'Borderlands'], errors='ignore')

    df = df.rename(columns={
        "Positive": "sentiment",
        "im getting on borderlands and i will murder you all ,": "text"
    })

    df = df[df['sentiment'] != 'Irrelevant']
    df = df.dropna()
    df = df.drop_duplicates(subset='text')

    df['sentiment'] = df['sentiment'].str.lower()

    # Text pipeline (vectorized)
    df['text'] = vectorized_clean_text(df['text'])
    df = df[df['text'] != '']

    df['text'] = remove_punctuation_series(df['text'])
    df['text'] = remove_stopwords_series(df['text'])
    df['text'] = lemmatize_series(df['text'])

    return df

In [ ]:
cleaned_df = optimized_preprocessing_pipeline(df)

In [ ]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 57259 entries, 0 to 74680
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  57259 non-null  object
 1   text       57259 non-null  object
dtypes: object(2)
memory usage: 1.3+ MB


In [ ]:
cleaned_df.isna().sum()

,0
sentiment,0
text,0


In [ ]:
!pip install dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.4/263.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

In [ ]:
import dagshub
import mlflow
dagshub.init(repo_owner='Aryanupadhyay23', repo_name='Twitter-Sentiment-Analysis-MLOps', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d736ad43-8ec3-4b86-bd38-5b3e82d0dead&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a26fce31e50251e0ed729780f03784cda0ed74b71d20f476b4416428c0eb5a0a




Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.head()

,sentiment,text
0,positive,come border kill
1,positive,im get borderland kill
2,positive,im come borderland murder
3,positive,im get borderland 2 murder
4,positive,im get borderland murder


In [ ]:
mapping = {
    "negative": -1,
    "neutral": 0,
    "positive": 1
}

df["sentiment"] = df["sentiment"].map(mapping)

In [ ]:
vectorizer = CountVectorizer(max_features=4000)

In [ ]:
X = vectorizer.fit_transform(df["text"])
y = df["sentiment"]

In [ ]:
X[0]

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3 stored elements and shape (1, 4000)>

In [ ]:
X.shape

(57259, 4000)

In [ ]:
y

,sentiment
0,1
1,1
2,1
3,1
4,1
...,...
74676,1
74677,1
74678,1
74679,1


In [ ]:
y.shape

(57259,)

In [ ]:
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow")

In [ ]:
mlflow.set_experiment("Baseline-Logistic-Regression")

<Experiment: artifact_location='mlflow-artifacts:/7bac28a644284eceabcf236f8a29bdb6', creation_time=1771653924109, experiment_id='1', last_update_time=1771653924109, lifecycle_stage='active', name='Baseline-Logistic-Regression', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [ ]:
X = df["text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
with mlflow.start_run():

    # Tags
    mlflow.set_tag("mlflow.runName", "logistic_regression_baseline")
    mlflow.set_tag("mlflow.user", "aryan")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("model_type", "logistic_regression")

    # Vectorization
    vectorizer = CountVectorizer(max_features=5000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_max_features", 5000)
    mlflow.log_param("vocab_size", len(vectorizer.vocabulary_))

    # Model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_vec, y_train)

    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("solver", model.solver)

    # Predictions
    y_train_pred = model.predict(X_train_vec)
    y_test_pred = model.predict(X_test_vec)

    # Metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    report_dict = classification_report(y_test, y_test_pred, output_dict=True)
    report_text = classification_report(y_test, y_test_pred)

    # Print classification report to console
    print("\nClassification Report:\n")
    print(report_text)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)

    mlflow.log_metric("precision_macro", report_dict["macro avg"]["precision"])
    mlflow.log_metric("recall_macro", report_dict["macro avg"]["recall"])
    mlflow.log_metric("f1_macro", report_dict["macro avg"]["f1-score"])

    mlflow.log_metric("precision_weighted", report_dict["weighted avg"]["precision"])
    mlflow.log_metric("recall_weighted", report_dict["weighted avg"]["recall"])
    mlflow.log_metric("f1_weighted", report_dict["weighted avg"]["f1-score"])

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.close()

    # Save Classification Report
    with open("classification_report.txt", "w") as f:
        f.write(report_text)

    mlflow.log_artifact("classification_report.txt")

    # Additional Model Info
    mlflow.log_param("num_train_samples", X_train_vec.shape[0])
    mlflow.log_param("num_test_samples", X_test_vec.shape[0])
    mlflow.log_param("num_features", X_train_vec.shape[1])
    mlflow.log_param("coef_shape", model.coef_.shape)

    # Log Model
    mlflow.sklearn.log_model(model, "logistic_regression_model")

print("Baseline run logged successfully.")


Classification Report:

              precision    recall  f1-score   support

          -1       0.78      0.82      0.80      4231
           0       0.76      0.67      0.71      3407
           1       0.76      0.80      0.78      3814

    accuracy                           0.77     11452
   macro avg       0.77      0.76      0.76     11452
weighted avg       0.77      0.77      0.77     11452



2026/02/21 06:34:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 06:34:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logistic_regression_baseline at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/1/runs/70bcffded74645bb99d07b5660c0e0ee
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/1
Baseline run logged successfully.


In [ ]:
cleaned_df.to_csv('twitter_cleaned.csv', index=False)